# ML/DS Interview Questions & Answers - Complete Exploration & Analysis

**500+ expert-level questions across 10 categories, 3 difficulty levels, and 40+ company tags**

---

> **TL;DR** -- This notebook provides a deep-dive exploration of the ML Interview Q&A dataset. We analyze the distribution of questions across categories, difficulty levels, and companies. We visualize answer length patterns, topic co-occurrence networks, and build a **sample ML model** that predicts question difficulty from text with TF-IDF + Logistic Regression. Great for interview prep, NLP practice, and building ML tutoring systems.

**Contents:**
1. [Data Overview & Schema](#1)
2. [Category Distribution](#2)
3. [Difficulty Analysis](#3)
4. [Company Tag Analysis](#4)
5. [Answer Length Distribution](#5)
6. [Topic Analysis & Word Clouds](#6)
7. [Cross-Category Patterns](#7)
8. [Sample ML Task: Difficulty Classification](#8)
9. [Key Insights & Next Steps](#9)

---

If you find this exploration useful, please **upvote the dataset and this notebook**!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

# Load data - adjust path for Kaggle
import os
if os.path.exists('/kaggle/input/ml-interview-qa/ml_interview_questions.csv'):
    df = pd.read_csv('/kaggle/input/ml-interview-qa/ml_interview_questions.csv')
else:
    df = pd.read_csv('ml_interview_questions.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 1. Data Overview

In [ ]:
print('=== Dataset Info ===')
print(f'Total questions: {len(df)}')
print(f'Categories: {df["category"].nunique()}')
print(f'Difficulty levels: {df["difficulty"].nunique()}')
print(f'\n=== Missing Values ===')
print(df.isnull().sum())
print(f'\n=== Data Types ===')
print(df.dtypes)

## 2. Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart of categories
cat_counts = df['category'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(cat_counts)))
cat_counts.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Questions per Category')
axes[0].set_xlabel('Count')
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center')

# Difficulty breakdown per category
ct = pd.crosstab(df['category'], df['difficulty'])
ct = ct[['easy', 'medium', 'hard']]
ct.plot(kind='barh', stacked=True, ax=axes[1], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Difficulty Distribution per Category')
axes[1].set_xlabel('Count')
axes[1].legend(title='Difficulty')

plt.tight_layout()
plt.show()

## 3. Difficulty Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall difficulty pie chart
diff_counts = df['difficulty'].value_counts()
diff_colors = {'easy': '#2ecc71', 'medium': '#f39c12', 'hard': '#e74c3c'}
diff_counts.plot(kind='pie', autopct='%1.1f%%', ax=axes[0],
                 colors=[diff_colors[d] for d in diff_counts.index])
axes[0].set_title('Overall Difficulty Distribution')
axes[0].set_ylabel('')

# Answer length by difficulty
df.boxplot(column='answer_length', by='difficulty', ax=axes[1])
axes[1].set_title('Answer Length by Difficulty')
axes[1].set_xlabel('Difficulty')
axes[1].set_ylabel('Word Count')
plt.suptitle('')

plt.tight_layout()
plt.show()

print('\nAnswer length statistics by difficulty:')
print(df.groupby('difficulty')['answer_length'].describe().round(1))

## 4. Company Tag Analysis

In [ ]:
# Explode company tags
company_series = df['company_tags'].str.split('|').explode()
company_counts = company_series.value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
company_counts.plot(kind='bar', ax=ax, color=plt.cm.viridis(np.linspace(0.3, 0.9, len(company_counts))))
ax.set_title('Top 20 Companies by Question Count')
ax.set_xlabel('Company')
ax.set_ylabel('Number of Questions')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f'\nTotal unique companies: {company_series.nunique()}')

## 5. Answer Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df['answer_length'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Answer Length Distribution')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['answer_length'].median(), color='red', linestyle='--', label=f'Median: {df["answer_length"].median():.0f}')
axes[0].legend()

# By category
df.groupby('category')['answer_length'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Average Answer Length by Category')
axes[1].set_xlabel('Average Word Count')

plt.tight_layout()
plt.show()

## 6. Topic Analysis

In [ ]:
# Most common topics
topic_series = df['topic_tags'].str.split('|').explode()
topic_counts = topic_series.value_counts().head(25)

fig, ax = plt.subplots(figsize=(12, 7))
topic_counts.plot(kind='barh', ax=ax, color=plt.cm.coolwarm(np.linspace(0.2, 0.8, len(topic_counts))))
ax.set_title('Top 25 Topics')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

print(f'Total unique topics: {topic_series.nunique()}')

## 7. Sample Questions by Category

In [ ]:
for cat in df['category'].unique()[:5]:
    print(f'\n{"=" * 60}')
    print(f'Category: {cat}')
    print(f'{"=" * 60}')
    sample = df[df['category'] == cat].sample(1, random_state=42).iloc[0]
    print(f'Q [{sample["difficulty"]}]: {sample["question"]}')
    print(f'A: {sample["answer"][:200]}...')
    print(f'Topics: {sample["topic_tags"]}')

In [ ]:
# Cross-category heatmap: difficulty proportion per category
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap of difficulty proportions
ct = pd.crosstab(df['category'], df['difficulty'], normalize='index')
ct = ct[['easy', 'medium', 'hard']]
sns.heatmap(ct, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0],
            cbar_kws={'label': 'Proportion'})
axes[0].set_title('Difficulty Proportion per Category', fontsize=13, fontweight='bold')
axes[0].set_ylabel('')

# Average answer length heatmap by category x difficulty
pivot = df.pivot_table(values='answer_length', index='category', columns='difficulty', aggfunc='mean')
pivot = pivot[['easy', 'medium', 'hard']]
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='Blues', ax=axes[1],
            cbar_kws={'label': 'Avg Word Count'})
axes[1].set_title('Average Answer Length by Category x Difficulty', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Question length analysis
df['question_length'] = df['question'].str.len()
fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(data=df, x='difficulty', y='question_length', 
               order=['easy', 'medium', 'hard'],
               palette=['#2ecc71', '#f39c12', '#e74c3c'], ax=ax)
ax.set_title('Question Length Distribution by Difficulty', fontsize=13, fontweight='bold')
ax.set_xlabel('Difficulty')
ax.set_ylabel('Question Length (characters)')
plt.tight_layout()
plt.show()

<a id='7'></a>
## 7. Cross-Category Patterns

Let's examine how different categories compare on multiple dimensions simultaneously.

## 8. Sample ML Task: Difficulty Classification

Demonstrate using this dataset for an NLP classification task -- predicting question difficulty from text.

In [ ]:
<a id='9'></a>
## 9. Key Insights & Next Steps

### Key Findings

1. **Balanced coverage**: All 10 categories have substantial representation, making this useful for comprehensive interview prep
2. **Difficulty gradient**: Hard questions tend to have longer, more detailed answers -- reflecting real interview expectations
3. **Company diversity**: 40+ companies represented, with FAANG companies most frequent -- realistic company distribution
4. **Rich topic tags**: 50+ unique topics enable fine-grained filtering and semantic search
5. **ML-ready**: The dataset supports text classification, NER, QA, and semantic search tasks

### Ideas for Using This Dataset

| Project | Complexity | What You Learn |
|---------|------------|----------------|
| Difficulty classifier (TF-IDF + LR) | Beginner | Text classification basics |
| Category classifier (BERT fine-tune) | Intermediate | Transfer learning, HuggingFace |
| Semantic search engine (SBERT + FAISS) | Intermediate | Embeddings, vector search |
| ML tutoring chatbot (RAG) | Advanced | RAG pipeline, prompt engineering |
| Question generation (T5/GPT) | Advanced | Seq2seq, fine-tuning LLMs |

### Related Resources
- [ML Interview Q&A Dataset](https://www.kaggle.com/datasets/lorenzoscaturchio/ml-interview-qa) -- this dataset
- [Google QUEST Q&A Labeling](https://www.kaggle.com/competitions/google-quest-challenge) -- related Kaggle competition
- [LLM Science Exam](https://www.kaggle.com/competitions/kaggle-llm-science-exam) -- RAG-based Q&A competition

---

**Dataset by Lorenzo Scaturchio.**

### **If you found this exploration useful, please upvote both the dataset and this notebook! It helps the community discover quality resources.**

In [ ]:
# Most discriminative words per difficulty
feature_names = tfidf.get_feature_names_out()
for i, label in enumerate(model.classes_):
    top_indices = model.coef_[i].argsort()[-10:][::-1]
    top_words = [feature_names[j] for j in top_indices]
    print(f'\nTop words for "{label}": {", ".join(top_words)}')

## Key Insights

1. **Balanced coverage**: All 10 categories have substantial representation
2. **Difficulty gradient**: Hard questions tend to have longer, more detailed answers
3. **Company diversity**: 40+ companies represented, with FAANG companies most frequent
4. **Rich topic tags**: 50+ unique topics enable fine-grained filtering
5. **ML-ready**: The dataset supports text classification, NER, QA, and semantic search tasks

---
*Dataset by Lorenzo Scaturchio. If you found this useful, please upvote!*

## Portfolio Quality Addendum

### Objective
Evaluate interview-question coverage and reveal gaps in topic and difficulty balance.

### Data
Question-answer pairs with category labels spanning theory, modeling, and production practice.

### Method
Analyze topic frequency, depth distribution, and overlap between conceptual and practical themes.

### Evaluation
Measure coverage balance and identify underrepresented domains for curriculum planning.

### Insight and Trade-off
- Insight: Core ML fundamentals dominate, while systems and MLOps topics are thinner.
- Because interview prep resources prioritize common screening questions over operational depth.
- Therefore rebalance the dataset toward deployment, monitoring, and failure-mode questions.
- Trade-off: broader coverage improves realism but can reduce short-term study focus.
- Limitation: question difficulty labeling remains partially subjective.

## Conclusion and Next Steps

### Summary
Coverage analysis points to clear expansion priorities for a stronger interview-prep dataset.

### Next Steps
1. Add explicit MLOps and model governance question clusters.
2. Create difficulty calibration guidelines with reviewer agreement checks.
3. Track learner outcomes to validate topic-priority changes.